# White rust pixel map – XGBoost, task C: severity score (0–9)
UAV hyperspectral image of 24 February 2026 · model trained on the 2025-26 ASD leaf spectra (leave-one-replication-out evaluation)

| Item | This notebook |
|---|---|
| Model | **XGBoost** (machine learning), file `C_XGBoost.npz` |
| Why this model | best overall and best ML on the leaves |
| Input given to the model | direct: direct UAV reflectance |
| Features (computed from each pixel spectrum) | FD1090, FD1060, FD870, FD595, MCARI, Dr |
| Accuracy on the 48 UAV plot spectra | R² -0.113 |
| Output rasters | `C_XGBoost` (predicted severity score 0–9) – ENVI `.dat/.hdr`, GeoTIFF (if `rasterio` is installed) and a PNG map |

On the 48 UAV plots this model was below chance, so the map is **not validated** and must not be read as severity – it shows what the leaf model predicts.

**Steps.** 1 settings · 2 libraries and image helpers · 3 reference data · features · load the model · self-check on the 48 plot means · open the image and crop mask · pixel loop · map · per-plot accuracy · outputs.

**Requirements.** Python 3.9+, `numpy`, `scipy`, `pandas`, `matplotlib`; optional `rasterio` for GeoTIFF. The model is stored as plain arrays and predicted with NumPy exactly as scikit-learn/xgboost do, so their versions do not matter.
**Time.** Run first with `SPATIAL_STEP = 4` (preview, about 1/16 of the pixels), then `SPATIAL_STEP = 1` for the final map.

## 1 Settings
Change the four paths to your folders. Everything else can stay as it is.

In [ ]:
# ============================ SETTINGS ============================
HDR_FILE  = r'D:\Coalighned\24_02_2026\subset_200526.hdr'          # ENVI header of the 24 Feb image (the data file 'subset_200526' must be next to it)
DATA_NAME = 'subset_200526'                                         # name of the image data file
ROI_FILE  = r'D:\Coalighned\24_02_2026\48_plot_roi_new.xml'        # 48 plot ROIs, ROI #n = plot n (None = skip the per-plot accuracy)
MODEL_FILE = r'D:\Coalighned\white_rust_pixel_kit\models\C_XGBoost.npz'   # this model
PLOT_AVG_FILE = r'D:\Coalighned\24_02_2026\envi_plot_avg.txt'      # ENVI plot-average file for the self-check (None = skip)
OUT_NAME  = 'white_rust_pixel_map_24Feb2026_C_XGBoost'                  # results folder, created next to the image

SPATIAL_STEP = 4        # 4 = quick preview; 1 = final full-resolution map
BLOCK_ROWS   = 16       # image rows processed at a time (reduce if memory is short)
NDVI_MIN     = 0.6      # crop mask: NDVI(898, 684) > 0.6 as in the methodology
REFLECTANCE_SCALE = 'auto'  # 'auto' (divides by 10000 if the data look like 0-10000) or a number

## 2 Libraries and image helpers
The image is memory-mapped: only the rows being processed are read, so a 9 GB image runs in a few GB of memory. The ROI parser converts the plot polygons from map coordinates to image pixels.

In [ ]:
import os, re, json, time, math, warnings
import xml.etree.ElementTree as ET
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.signal import savgol_coeffs
warnings.filterwarnings('ignore', category=RuntimeWarning)
try:
    from IPython.display import Image as _Img, display as _display
    def show(path, width=900): _display(_Img(path, width=width))
except Exception:
    def show(path, width=900): print('saved', path)

assert os.path.exists(HDR_FILE), f'Header not found: {HDR_FILE}'
SRC = os.path.dirname(HDR_FILE); _base = os.path.splitext(HDR_FILE)[0]
DATA_FILE = next((c for c in [os.path.join(SRC, DATA_NAME), _base, _base + '.dat', _base + '.img', _base + '.bil', _base + '.raw']
                  if os.path.isfile(c) and not c.lower().endswith('.hdr')), None)
if DATA_FILE is None:
    raise FileNotFoundError(f'Image data file not found next to {HDR_FILE}. Files there: {os.listdir(SRC)}')
OUT = os.path.join(SRC, OUT_NAME); os.makedirs(OUT, exist_ok=True)
assert os.path.exists(MODEL_FILE), f'Model file not found: {MODEL_FILE}'

ENVI_DTYPES = {1: np.uint8, 2: np.int16, 3: np.int32, 4: np.float32, 5: np.float64, 12: np.uint16, 13: np.uint32, 14: np.int64, 15: np.uint64}
def read_envi_header(path):
    txt = open(path, encoding='latin-1').read(); hdr = {}
    for m in re.finditer(r'^\s*([^=\n{}]+?)\s*=\s*(\{[^}]*\}|[^\n]*)', txt, re.M):
        v = m.group(2).strip(); hdr[m.group(1).strip().lower()] = v[1:-1].strip() if v.startswith('{') else v
    return hdr
class EnviCube:
    # memory-mapped input image; reads only what is needed
    def __init__(self, path, hdr):
        self.ns, self.nl, self.nb = int(hdr['samples']), int(hdr['lines']), int(hdr['bands'])
        dt = np.dtype(ENVI_DTYPES[int(hdr['data type'])])
        if int(hdr.get('byte order', '0')) == 1: dt = dt.newbyteorder('>')
        self.il = hdr.get('interleave', 'bsq').strip().lower()
        shape = {'bil': (self.nl, self.nb, self.ns), 'bsq': (self.nb, self.nl, self.ns), 'bip': (self.nl, self.ns, self.nb)}[self.il]
        off = int(hdr.get('header offset', '0'))
        if os.path.getsize(path) < off + int(np.prod(shape)) * dt.itemsize: raise ValueError('data file is smaller than the header says')
        self.mm = np.memmap(path, dtype=dt, mode='r', offset=off, shape=shape)
    def rows(self, r0, r1, step):
        # rows r0..r1 (full-res, every step-th row and column) -> (rows, cols, bands) float32
        ys, xs = slice(r0, r1, step), slice(0, self.ns, step)
        if self.il == 'bil': a = np.asarray(self.mm[ys, :, xs]).transpose(0, 2, 1)
        elif self.il == 'bsq': a = np.asarray(self.mm[:, ys, xs]).transpose(1, 2, 0)
        else: a = np.asarray(self.mm[ys, xs])
        return a.astype(np.float32)
class Geo:
    def __init__(self, hdr):
        mi = [s.strip() for s in hdr['map info'].split(',')]; self.items, self.proj = mi, mi[0]
        self.refx, self.refy, self.e0, self.n0, self.dx, self.dy = map(float, mi[1:7]); self.csys = hdr.get('coordinate system string', '').strip()
        g = self.proj.lower().startswith('geographic')
        self.px_w = self.dx * (111320.0 * np.cos(np.radians(self.n0)) if g else 1.0); self.px_h = self.dy * (110574.0 if g else 1.0)
    def to_file(self, x, y):
        x, y = np.asarray(x, float), np.asarray(y, float); return (x - self.e0) / self.dx + (self.refx - 1), (self.n0 - y) / self.dy + (self.refy - 1)
    def to_map(self, fx, fy):
        fx, fy = np.asarray(fx, float), np.asarray(fy, float); return self.e0 + (fx - (self.refx - 1)) * self.dx, self.n0 - (fy - (self.refy - 1)) * self.dy
    def map_info(self, step=1):
        e, n = self.to_map(0.5 - 0.5 * step, 0.5 - 0.5 * step)          # output pixel centres = sampled input pixel centres
        return '{' + ', '.join([self.proj, '1.0000', '1.0000', f'{float(e):.10f}', f'{float(n):.10f}', repr(self.dx * step), repr(self.dy * step)] + self.items[7:]) + '}'
def write_envi(path, arr2d, band_name, dtype_code, extra=None):
    arr2d[:, None, :].astype({4: '<f4', 1: 'u1'}[dtype_code]).tofile(path)
    L = ['ENVI', f'description = {{{os.path.basename(path)}}}', f'samples = {arr2d.shape[1]}', f'lines   = {arr2d.shape[0]}', 'bands   = 1', 'header offset = 0',
         'file type = ENVI Standard', f'data type = {dtype_code}', 'interleave = bil', 'byte order = 0', f'map info = {geo.map_info(SPATIAL_STEP)}', 'band names = {' + band_name + '}']
    if geo.csys: L.append('coordinate system string = {' + geo.csys + '}')
    for k, v in (extra or {}).items(): L.append(f'{k} = {v}')
    open(os.path.splitext(path)[0] + '.hdr', 'w').write('\n'.join(L) + '\n')
def write_tif(path, arr2d, nodata):
    try:
        import rasterio
        from rasterio.transform import from_origin
    except ImportError:
        return False
    e0, n0 = geo.to_map(0.5 - 0.5 * SPATIAL_STEP, 0.5 - 0.5 * SPATIAL_STEP)
    with rasterio.open(path, 'w', driver='GTiff', height=arr2d.shape[0], width=arr2d.shape[1], count=1, dtype=str(arr2d.dtype), crs='EPSG:4326',
                       transform=from_origin(float(e0), float(n0), geo.dx * SPATIAL_STEP, geo.dy * SPATIAL_STEP), nodata=nodata) as dst:
        dst.write(arr2d, 1)
    return True
def read_roi_xml(path):
    # ENVI ROI .xml -> {plot number: [polygons in full-res file coordinates]}
    plots = {}
    for i, reg in enumerate(ET.parse(path).getroot().iter('Region')):
        m = re.search(r'(\d+)\D*$', reg.get('name', '')); pno = int(m.group(1)) if m else i + 1; polys = []
        for gd in reg.iter('GeometryDef'):
            cs = (gd.findtext('CoordSysStr') or 'none').strip()
            for poly in gd.iter('Polygon'):
                c = poly.find('.//Exterior//Coordinates')
                if c is None or not (c.text or '').strip(): continue
                v = np.array(c.text.split(), float).reshape(-1, 2)
                fx, fy = (v[:, 0], v[:, 1]) if cs.lower() == 'none' else geo.to_file(v[:, 0], v[:, 1])
                polys.append(np.column_stack([fx, fy]))
        if polys: plots[pno] = polys
    return dict(sorted(plots.items()))
hdr = read_envi_header(HDR_FILE); cube = EnviCube(DATA_FILE, hdr); geo = Geo(hdr)
WL = np.array([float(t) for t in hdr['wavelength'].split(',')])
print(f'image {cube.ns} x {cube.nl} px, {cube.nb} bands ({cube.il}), pixel {geo.px_w * 100:.2f} x {geo.px_h * 100:.2f} cm, {WL[0]:.1f}-{WL[-1]:.1f} nm')

## 3 Reference data (embedded)
`ASD_HEALTHY` = mean of the healthy ASD leaves of 24 Feb at the image wavelengths, `UAV_HEALTHY` = mean of the 12 sprayed plots of this image; their ratio is the healthy-reference gain used by the *corrected* and *normalised* inputs (empirical-line idea; Smith and Milton, 1999; brightness normalisation: Barnes et al., 1989). `TRUTH` = ASD leaf scores of 24 Feb per plot. `EXPECTED` = this model's prediction for the 48 plot-mean spectra (self-check).

In [ ]:
REF_WL = np.array([399.375, 401.132996, 402.890991, 404.648987, 406.407013, 408.165009, 409.923004, 411.679993, 413.437988, 415.196014, 416.95401, 418.712006, 420.470001, 422.227997, 423.985992, 425.743988, 427.502014, 429.26001, 431.016998, 432.774994, 434.53299, 436.290985, 438.049011, 439.807007, 441.565002, 443.322998, 445.080994, 446.838989, 448.596985, 450.354004, 452.112, 453.869995, 455.627991, 457.385986, 459.144012, 460.902008, 462.660004, 464.417999, 466.175995, 467.93399, 469.691986, 471.449005, 473.207001, 474.964996, 476.722992, 478.480988, 480.239014, 481.997009, 483.755005, 485.513, 487.270996, 489.028992, 490.786011, 492.544006, 494.302002, 496.059998, 497.817993, 499.575989, 501.334015, 503.09201, 504.850006, 506.608002, 508.365997, 510.123993, 511.881012, 513.638977, 515.396973, 517.155029, 518.913025, 520.671021, 522.429016, 524.187012, 525.945007, 527.703003, 529.460999, 531.218018, 532.976013, 534.734009, 536.492004, 538.25, 540.007996, 541.765991, 543.523987, 545.281982, 547.039978, 548.797974, 550.55603, 552.312988, 554.070984, 555.828979, 557.586975, 559.344971, 561.103027, 562.861023, 564.619019, 566.377014, 568.13501, 569.893005, 571.650024, 573.40802, 575.166016, 576.924011, 578.682007, 580.440002, 582.197998, 583.955994, 585.713989, 587.471985, 589.22998, 590.987, 592.744995, 594.502991, 596.260986, 598.018982, 599.776978, 601.534973, 603.29303, 605.051025, 606.809021, 608.567017, 610.325012, 612.08197, 613.840027, 615.598022, 617.356018, 619.114014, 620.872009, 622.630005, 624.388, 626.145996, 627.903992, 629.661987, 631.419006, 633.177002, 634.934998, 636.692993, 638.450989, 640.208984, 641.96698, 643.724976, 645.482971, 647.241028, 648.999023, 650.757019, 652.513977, 654.271973, 656.030029, 657.788025, 659.546021, 661.304016, 663.062012, 664.820007, 666.578003, 668.335999, 670.093994, 671.851013, 673.609009, 675.367004, 677.125, 678.882996, 680.640991, 682.398987, 684.156982, 685.914978, 687.672974, 689.43103, 691.187988, 692.945984, 694.703979, 696.461975, 698.219971, 699.978027, 701.736023, 703.494019, 705.252014, 707.01001, 708.768005, 710.526001, 712.28302, 714.041016, 715.799011, 717.557007, 719.315002, 721.072998, 722.830994, 724.588989, 726.346985, 728.10498, 729.862976, 731.619995, 733.377991, 735.135986, 736.893982, 738.651978, 740.409973, 742.16803, 743.926025, 745.684021, 747.442017, 749.200012, 750.958008, 752.715027, 754.473022, 756.231018, 757.989014, 759.747009, 761.505005, 763.263, 765.020996, 766.778992, 768.536987, 770.294983, 772.052002, 773.809998, 775.567993, 777.325989, 779.083984, 780.84198, 782.599976, 784.357971, 786.116028, 787.874023, 789.632019, 791.388977, 793.146973, 794.905029, 796.663025, 798.421021, 800.179016, 801.937012, 803.695007, 805.453003, 807.210999, 808.968994, 810.72699, 812.484009, 814.242004, 816, 817.757996, 819.515991, 821.273987, 823.031982, 824.789978, 826.547974, 828.30603, 830.064026, 831.820984, 833.578979, 835.336975, 837.094971, 838.853027, 840.611023, 842.369019, 844.127014, 845.88501, 847.643005, 849.401001, 851.158997, 852.916016, 854.674011, 856.432007, 858.190002, 859.947998, 861.705994, 863.463989, 865.221985, 866.97998, 868.737976, 870.495972, 872.252991, 874.010986, 875.768982, 877.526978, 879.284973, 881.04303, 882.801025, 884.559021, 886.317017, 888.075012, 889.833008, 891.591003, 893.348022, 895.106018, 896.864014, 898.622009, 916.200989, 917.958984, 919.71698, 921.372009, 927.372986, 933.372986, 939.374023, 945.375, 951.375, 957.375977, 963.377014, 969.377991, 975.377991, 981.379028, 987.380005, 993.380005, 999.380981, 1005.38, 1011.38, 1017.38, 1023.38, 1029.38, 1035.39001, 1041.39001, 1047.39001, 1053.39001, 1059.39001, 1065.39001, 1071.39001, 1077.39001, 1083.39001, 1089.39001, 1095.39001, 1101.39001, 1107.39001, 1113.39001, 1119.40002, 1125.40002, 1131.40002, 1137.40002, 1143.40002, 1149.40002, 1155.40002, 1161.40002, 1167.40002, 1173.40002, 1179.40002, 1185.40002, 1191.40002, 1197.40002, 1203.41003, 1209.41003, 1215.41003, 1221.41003, 1227.41003, 1233.41003, 1239.41003, 1245.41003, 1251.41003, 1257.41003, 1263.41003, 1269.41003, 1275.41003, 1281.41003, 1287.42004, 1293.42004, 1299.42004, 1305.42004, 1311.42004, 1317.42004, 1323.42004, 1329.42004, 1461.43994, 1467.43994, 1473.43994, 1479.43994, 1485.43994, 1491.43994, 1497.43994, 1503.43994, 1509.43994, 1515.43994, 1521.43994, 1527.43994, 1533.43994, 1539.44995, 1545.44995, 1551.44995, 1557.44995, 1563.44995, 1569.44995, 1575.44995, 1581.44995, 1587.44995, 1593.44995, 1599.44995, 1605.44995, 1611.44995, 1617.44995, 1623.45996, 1629.45996, 1635.45996, 1641.45996, 1647.45996, 1653.45996, 1659.45996, 1665.45996, 1671.45996, 1677.45996, 1683.45996, 1689.45996, 1695.45996, 1701.45996, 1707.46997, 1713.46997, 1719.46997, 1725.46997, 1731.46997, 1737.46997, 1743.46997, 1749.46997, 1755.46997, 1761.46997, 1767.46997, 1773.46997, 1779.46997, 1971.5, 1977.5, 1983.5, 1989.5, 1995.5, 2001.5, 2007.5, 2013.5, 2019.5, 2025.5, 2031.5, 2037.5, 2043.51001, 2049.51001, 2055.51001, 2061.51001, 2067.51001, 2073.51001, 2079.51001, 2085.51001, 2091.51001, 2097.51001, 2103.51001, 2109.51001, 2115.51001, 2121.51001, 2127.52002, 2133.52002, 2139.52002, 2145.52002, 2151.52002, 2157.52002, 2163.52002, 2169.52002, 2175.52002, 2181.52002, 2187.52002, 2193.52002, 2199.52002, 2205.52002, 2211.53003, 2217.53003, 2223.53003, 2229.53003, 2235.53003, 2241.53003, 2247.53003, 2253.53003, 2259.53003, 2265.53003, 2271.53003, 2277.53003, 2283.53003, 2289.53003, 2295.54004, 2301.54004, 2307.54004, 2313.54004, 2319.54004, 2325.54004, 2331.54004, 2337.54004, 2343.54004, 2349.54004, 2355.54004, 2361.54004, 2367.54004, 2373.54004, 2379.55005, 2385.55005, 2391.55005, 2397.55005])
ASD_HEALTHY = np.array([0.2801424462, 0.2779653537, 0.2770459872, 0.2767273765, 0.2724106125, 0.2691869179, 0.2666070477, 0.2662811698, 0.2640439365, 0.2614161355, 0.2584511791, 0.2558659918, 0.252921003, 0.2500232907, 0.2470388745, 0.2439695283, 0.2410331271, 0.2372346909, 0.2335354703, 0.2310966937, 0.2273987308, 0.2244191144, 0.2207646543, 0.2180943533, 0.2152876755, 0.2124442901, 0.2103523876, 0.2078841549, 0.205567424, 0.2035639439, 0.2010999906, 0.1986544038, 0.1962185855, 0.1938433226, 0.1911036507, 0.1887047455, 0.1862789251, 0.1838556643, 0.1814299236, 0.1788884079, 0.1765870084, 0.1745310455, 0.1724275889, 0.1704791188, 0.1685358487, 0.1664482431, 0.164833341, 0.1629113156, 0.1612406608, 0.1597603278, 0.158331235, 0.157215873, 0.1561415129, 0.1552601854, 0.1548869617, 0.1546245456, 0.1546644734, 0.1549757714, 0.1560308809, 0.1572299406, 0.1590616314, 0.1613940426, 0.16447533, 0.1683146806, 0.1730668605, 0.1785615686, 0.1849773906, 0.1922807558, 0.2004031726, 0.2089309438, 0.217743198, 0.2264223891, 0.2346209605, 0.2420865836, 0.248468632, 0.2537962573, 0.2580903812, 0.2613027402, 0.2635570942, 0.2650315641, 0.2658656364, 0.2663628226, 0.2667432325, 0.2670606312, 0.2672963845, 0.2673435952, 0.2669638093, 0.2660348816, 0.2641703301, 0.2613535001, 0.2576685, 0.2533068989, 0.2485391187, 0.24333415, 0.2376756059, 0.2316005931, 0.2252014813, 0.2185310516, 0.2117699212, 0.2051747187, 0.1989710825, 0.1933888739, 0.1885561934, 0.1841439004, 0.1801805688, 0.1765380437, 0.1733091139, 0.1704388223, 0.1679654918, 0.1658733794, 0.1640720295, 0.1624854006, 0.1610561101, 0.1596383156, 0.1580919504, 0.1563603433, 0.1542877303, 0.1519392644, 0.1492173354, 0.1463000605, 0.1432484143, 0.140181812, 0.1372950266, 0.1345611894, 0.1321567857, 0.1300698342, 0.1283471613, 0.1269573688, 0.1258242425, 0.1249811424, 0.1243400913, 0.1238400879, 0.1233163534, 0.1226837855, 0.121755371, 0.1204144326, 0.1187915136, 0.1169129459, 0.1148271764, 0.1127028855, 0.1104756905, 0.1083272586, 0.1063881048, 0.1047091297, 0.1031836934, 0.1016900119, 0.1000380443, 0.09819197981, 0.09626704333, 0.09437766325, 0.09267462287, 0.09127186081, 0.09024796118, 0.08955409158, 0.08917557833, 0.08909465553, 0.08926054989, 0.08959964262, 0.09018555206, 0.09099060631, 0.09208383427, 0.09346431269, 0.09524725594, 0.0976544383, 0.1011726582, 0.1063978624, 0.1139260472, 0.1242745065, 0.137864584, 0.1543321779, 0.1732972371, 0.1941942999, 0.2165455377, 0.239613005, 0.2630258291, 0.2865433902, 0.3100210549, 0.3334766028, 0.3567596381, 0.3798594943, 0.4025120918, 0.4246402575, 0.4460716724, 0.4666567954, 0.4861559929, 0.5043677991, 0.5212204092, 0.5366808477, 0.5506012813, 0.5630071257, 0.5740868633, 0.5838765389, 0.592435795, 0.5997856022, 0.6061688409, 0.611567706, 0.616164028, 0.6198462913, 0.6229398266, 0.6254452966, 0.6273462852, 0.6288772382, 0.6300416613, 0.6309410531, 0.6315037919, 0.6319012564, 0.6322084413, 0.632377192, 0.6323873525, 0.6323337272, 0.6322255951, 0.6320236175, 0.6318117873, 0.6316223257, 0.6313868625, 0.6312344748, 0.6310703377, 0.6309014186, 0.6307738883, 0.6305807672, 0.6304664633, 0.6303427356, 0.6302312423, 0.6301273459, 0.6300199376, 0.6298883175, 0.6298317384, 0.6297639775, 0.6296716549, 0.6295622825, 0.6294761332, 0.6294123998, 0.629343421, 0.6293168995, 0.6291910453, 0.6290361701, 0.6289175414, 0.6288132952, 0.6287562897, 0.6286252506, 0.6284992684, 0.6284162865, 0.6282970247, 0.6280966693, 0.6278845456, 0.6275797274, 0.6273710122, 0.627282393, 0.6271597641, 0.6270349932, 0.626930705, 0.6268908252, 0.6268398305, 0.6267865117, 0.6267203507, 0.6266437383, 0.6266256483, 0.6265601579, 0.6264552923, 0.6264879895, 0.6265032985, 0.626399937, 0.6263183213, 0.6262692172, 0.6262114095, 0.6261351304, 0.6261445267, 0.6260343332, 0.6259806851, 0.6259043583, 0.6258319058, 0.6257365941, 0.625656762, 0.6255701921, 0.6255539102, 0.6254895404, 0.6253357917, 0.6252353176, 0.6252513587, 0.6251698402, 0.6251077451, 0.625050853, 0.6249454667, 0.6248772546, 0.6248762517, 0.6245077471, 0.6242841374, 0.6241030263, 0.6239030758, 0.6223692184, 0.6196699608, 0.6161244215, 0.6115083144, 0.6051487716, 0.5984380309, 0.5947664832, 0.5937437234, 0.593868761, 0.5950071041, 0.5970639472, 0.5996297318, 0.6025973619, 0.6030587577, 0.6060166444, 0.6090053442, 0.6118018697, 0.614229734, 0.6161009675, 0.6175464776, 0.6186414718, 0.6194746873, 0.6200185803, 0.6203076902, 0.6203330143, 0.6201365337, 0.6195229718, 0.6186536455, 0.6176814126, 0.6164241129, 0.6148956565, 0.6130726882, 0.6104819301, 0.6059055997, 0.5974698481, 0.5843495604, 0.5689988995, 0.555952204, 0.5481769802, 0.5449208614, 0.5439962518, 0.544181301, 0.5448416074, 0.5457987652, 0.5469680437, 0.5487328867, 0.5514065752, 0.5537279127, 0.5543115523, 0.5533640642, 0.5523517859, 0.5521783703, 0.5527459846, 0.5536593193, 0.5544612846, 0.5548770661, 0.5546087034, 0.5536104484, 0.5520053797, 0.549629383, 0.5463050507, 0.541802422, 0.5360184287, 0.5285724689, 0.5194528708, 0.5083794164, 0.4953816859, 0.4811191172, 0.1090909611, 0.1126489801, 0.1177516286, 0.1240744508, 0.1313763551, 0.1394959749, 0.148318622, 0.1577398625, 0.167641682, 0.1778947396, 0.1883583699, 0.1989811802, 0.2096555473, 0.2202641532, 0.2306818436, 0.2408853459, 0.2507348066, 0.2601358141, 0.2690583072, 0.2775331063, 0.2855097823, 0.2930319113, 0.3000516966, 0.3066306607, 0.3127576071, 0.3184336676, 0.3236878841, 0.3285415586, 0.3330870617, 0.3372175807, 0.3409417125, 0.3443120057, 0.3473277052, 0.349845745, 0.3519379215, 0.3537123412, 0.3554603633, 0.3579311185, 0.3611650302, 0.3643039038, 0.3673060804, 0.3691754971, 0.370627102, 0.372568889, 0.3709313022, 0.3608675317, 0.3463128562, 0.3348211235, 0.3272235608, 0.3208628445, 0.3129261879, 0.3025851745, 0.2930019481, 0.2864031973, 0.06223559481, 0.0629938025, 0.06391362873, 0.06503358005, 0.06635490244, 0.06789030375, 0.06963975129, 0.07158134482, 0.07372110813, 0.07607831053, 0.07866720489, 0.08142972971, 0.08435165988, 0.08739946297, 0.09059218852, 0.09401549546, 0.09769752756, 0.101563958, 0.1056054613, 0.1098064798, 0.1140701628, 0.1184226714, 0.1228828502, 0.1274127016, 0.131945463, 0.1364267764, 0.1406765012, 0.1445146759, 0.1479237749, 0.1508455066, 0.1533064883, 0.1554293456, 0.1573196411, 0.1590819537, 0.1607820906, 0.1624551012, 0.1640449266, 0.1654202134, 0.1665818779, 0.1674885732, 0.1681369508, 0.168500663, 0.1685993958, 0.1684959254, 0.1683410616, 0.168329379, 0.168869479, 0.1700597106, 0.1719740701, 0.1744995308, 0.1769696914, 0.1784591324, 0.1794759106, 0.1816914972, 0.186619898, 0.192753619, 0.1923190833, 0.1795356383, 0.1633689418, 0.1534444644, 0.149510604, 0.1499603135, 0.1518775345, 0.1480717692, 0.1383669039, 0.1307676167, 0.1266232725, 0.12330634, 0.1200807657, 0.116849162, 0.1135965695, 0.1104557403])
UAV_HEALTHY = np.array([0.01786958333, 0.01824916667, 0.01865383333, 0.01899975, 0.0192175, 0.01925233333, 0.01904925, 0.01870708333, 0.01830341667, 0.01793591667, 0.0176005, 0.01733541667, 0.01714841667, 0.01704333333, 0.01699458333, 0.01698458333, 0.01696033333, 0.01693458333, 0.01689041667, 0.01685466667, 0.01679458333, 0.01677116667, 0.01676575, 0.01682025, 0.01689375, 0.01698416667, 0.01706458333, 0.01717666667, 0.01731, 0.01747216667, 0.017614, 0.01776058333, 0.01786816667, 0.01796908333, 0.01802975, 0.01808583333, 0.01811583333, 0.018155, 0.01816716667, 0.018176, 0.01815216667, 0.01814608333, 0.01811683333, 0.01811591667, 0.01811575, 0.018173, 0.01822641667, 0.01831683333, 0.01838016667, 0.01846141667, 0.01854466667, 0.01869083333, 0.01886383333, 0.01914316667, 0.0194995, 0.01999025, 0.020553, 0.02121625, 0.02191041667, 0.022707, 0.02355358333, 0.02452875, 0.02558341667, 0.02682908333, 0.028192, 0.02971483333, 0.03129541667, 0.03297433333, 0.034695, 0.0365005, 0.0382985, 0.04013, 0.04186616667, 0.04358491667, 0.04517108333, 0.04664008333, 0.04789133333, 0.04902391667, 0.04997058333, 0.05079641667, 0.05146616667, 0.05205908333, 0.0525125, 0.0529475, 0.05328733333, 0.05359208333, 0.05379491667, 0.05394116667, 0.05391675, 0.05376333333, 0.05340091667, 0.05292466667, 0.05227733333, 0.05154583333, 0.050669, 0.04974725, 0.04871966667, 0.04769183333, 0.046622, 0.04559716667, 0.04456183333, 0.04363133333, 0.04275525, 0.04203341667, 0.04137625, 0.04084733333, 0.04036083333, 0.0399465, 0.03953591667, 0.03921383333, 0.03888783333, 0.03862791667, 0.0383565, 0.03815316667, 0.03789166667, 0.03764358333, 0.03730791667, 0.03692483333, 0.03645941667, 0.0359995, 0.03547708333, 0.034971, 0.03442741667, 0.0339465, 0.03345933333, 0.03308275, 0.03274141667, 0.03251966667, 0.0323285, 0.03222258333, 0.03210975, 0.03202025, 0.03187616667, 0.03173191667, 0.03150433333, 0.03126358333, 0.03093091667, 0.03058091667, 0.0301465, 0.02972075, 0.02925866667, 0.02883833333, 0.02839933333, 0.02801375, 0.02758241667, 0.02714358333, 0.02663883333, 0.02613925, 0.02559025, 0.02508566667, 0.02459291667, 0.02418775, 0.02383375, 0.02359358333, 0.02337966667, 0.02324491667, 0.02313025, 0.02311266667, 0.02315425, 0.02335041667, 0.02365983333, 0.02421791667, 0.02501916667, 0.02615016667, 0.02761, 0.02951708333, 0.03186475, 0.03480058333, 0.0382625, 0.04235325, 0.04692766667, 0.0520315, 0.05748108333, 0.06325966667, 0.06918, 0.07527241667, 0.081576, 0.0882255, 0.09517933333, 0.1025586667, 0.110077, 0.1176268333, 0.1251959167, 0.132893, 0.1403654167, 0.1476189167, 0.15449725, 0.1610945833, 0.1672806667, 0.17315225, 0.1784891667, 0.1833869167, 0.1879485833, 0.1922978333, 0.1962219167, 0.1997788333, 0.2028679167, 0.205574, 0.2074160833, 0.2089018333, 0.21023925, 0.212361, 0.21567775, 0.2202174167, 0.2233445, 0.2244345, 0.2236648333, 0.2213015833, 0.2178298333, 0.21437275, 0.211751, 0.2110959167, 0.2125753333, 0.21539475, 0.2171270833, 0.2180105, 0.21832725, 0.2183255833, 0.218387, 0.2185240833, 0.218733, 0.21891975, 0.218943, 0.2188821667, 0.21875725, 0.2186058333, 0.2183110833, 0.2179321667, 0.2176116667, 0.2173935, 0.2172420833, 0.2171883333, 0.217338, 0.2177443333, 0.2183655, 0.2189465, 0.2190189167, 0.2186199167, 0.2180638333, 0.2172909167, 0.21626525, 0.2154828333, 0.2151774167, 0.2151525, 0.21527125, 0.21532075, 0.21518275, 0.2149929167, 0.2148885, 0.2147624167, 0.2146066667, 0.2145094167, 0.2146885, 0.2149189167, 0.21528875, 0.2156349167, 0.21600675, 0.2161555833, 0.2158789167, 0.2155861667, 0.215294, 0.2150009167, 0.2151154167, 0.21544375, 0.2157681667, 0.2156671667, 0.2157010833, 0.2153201667, 0.2148168333, 0.21443775, 0.2141508333, 0.2140043333, 0.21400075, 0.21393275, 0.2136655833, 0.21310675, 0.21279425, 0.2122740833, 0.2121775833, 0.212321, 0.2135155, 0.2139616667, 0.21338025, 0.2111958333, 0.2101384167, 0.2082278333, 0.205455, 0.2033931667, 0.20175875, 0.2006719167, 0.1997563333, 0.1975608333, 0.1943045, 0.1901655833, 0.1874901667, 0.1840030833, 0.18083475, 0.1792470833, 0.178292, 0.1782279167, 0.1786015833, 0.1797715, 0.1811654167, 0.1827351667, 0.18455725, 0.186468, 0.1882451667, 0.1899151667, 0.1914273333, 0.1927998333, 0.1938561667, 0.1946763333, 0.1952229167, 0.1954671667, 0.1954400833, 0.1953338333, 0.1940515, 0.1919173333, 0.1904671667, 0.18971475, 0.189138, 0.1874116667, 0.18455375, 0.18086675, 0.1763464167, 0.17109125, 0.1646299167, 0.1584655, 0.15433, 0.1517856667, 0.1501198333, 0.1485311667, 0.14753175, 0.1472040833, 0.1471151667, 0.1472825, 0.1475795, 0.148068, 0.1487295833, 0.1495200833, 0.1500699167, 0.1505246667, 0.15106025, 0.1517270833, 0.1522425, 0.1527203333, 0.1528974167, 0.1527556667, 0.1520989167, 0.15093175, 0.1491018333, 0.1466416667, 0.1509860833, 0.14517025, 0.1321020833, 0.11422725, 0.09385, 0.07338425, 0.05510058333, 0.04090325, 0.03280475, 0.032358, 0.04107983333, 0.04201808333, 0.04324283333, 0.0443705, 0.04566575, 0.04705858333, 0.04871433333, 0.05034308333, 0.05220083333, 0.05402408333, 0.055933, 0.0577855, 0.05971316667, 0.06160683333, 0.06359583333, 0.06550008333, 0.06729375, 0.068953, 0.07060808333, 0.07207391667, 0.07358866667, 0.07473891667, 0.075683, 0.0763095, 0.07652008333, 0.076745, 0.07736316667, 0.07819225, 0.07930558333, 0.080483, 0.08180366667, 0.0826415, 0.083034, 0.08284566667, 0.08203933333, 0.08120083333, 0.08043991667, 0.07937925, 0.07832191667, 0.0772225, 0.07611666667, 0.07478775, 0.07339241667, 0.07183916667, 0.0744585, 0.07214358333, 0.06587225, 0.05694675, 0.04654483333, 0.03567541667, 0.02570683333, 0.01850508333, 0.01450066667, 0.01471558333, 0.020076, 0.02121966667, 0.021817, 0.022066, 0.02175416667, 0.02133391667, 0.02120891667, 0.02202883333, 0.02284716667, 0.02391416667, 0.02479041667, 0.02567475, 0.02626458333, 0.02685825, 0.02731375, 0.02802391667, 0.02869808333, 0.02967566667, 0.03051925, 0.03139725, 0.03201358333, 0.03267158333, 0.03319483333, 0.0337335, 0.03408783333, 0.03457333333, 0.03502275, 0.03558408333, 0.03587866667, 0.03628991667, 0.03655758333, 0.03705458333, 0.03742825, 0.03779508333, 0.03799808333, 0.0382855, 0.03837416667, 0.03840225, 0.03805925, 0.03756066667, 0.03678933333, 0.03605858333, 0.035045, 0.03408591667, 0.033236, 0.03262458333, 0.0320565, 0.03167641667, 0.03091483333, 0.03010308333, 0.02936158333, 0.02873158333, 0.02784708333, 0.02734158333, 0.02674925, 0.02641575, 0.02588966667, 0.0253955, 0.0245815, 0.02404041667, 0.02339866667, 0.0231525, 0.02273908333, 0.02251141667, 0.0220195, 0.02173625, 0.02127408333])
TRUTH = pd.DataFrame({'plot': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48], 'block': ['C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'R3', 'R3', 'R3', 'R3', 'R3', 'R3', 'R3', 'R3', 'R3', 'R3', 'R3', 'R3', 'R2', 'R2', 'R2', 'R2', 'R2', 'R2', 'R2', 'R2', 'R2', 'R2', 'R2', 'R2', 'R1', 'R1', 'R1', 'R1', 'R1', 'R1', 'R1', 'R1', 'R1', 'R1', 'R1', 'R1'], 'status': ['Healthy', 'Healthy', 'Healthy', 'Healthy', 'Healthy', 'Healthy', 'Healthy', 'Healthy', 'Healthy', 'Healthy', 'Healthy', 'Healthy', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased', 'Diseased'], 'variety': ['V2', 'V2', 'V3', 'V1', 'V2', 'V3', 'V1', 'V3', 'V1', 'V3', 'V2', 'V1', 'V2', 'V1', 'V1', 'V3', 'V2', 'V2', 'V2', 'V3', 'V3', 'V1', 'V3', 'V1', 'V3', 'V2', 'V3', 'V1', 'V3', 'V2', 'V1', 'V2', 'V3', 'V1', 'V2', 'V1', 'V1', 'V3', 'V2', 'V3', 'V1', 'V2', 'V1', 'V2', 'V2', 'V3', 'V1', 'V3'], 'sowing': ['D4', 'D2', 'D1', 'D3', 'D1', 'D4', 'D1', 'D2', 'D2', 'D3', 'D3', 'D4', 'D1', 'D4', 'D3', 'D2', 'D3', 'D4', 'D2', 'D4', 'D3', 'D2', 'D1', 'D1', 'D1', 'D4', 'D2', 'D4', 'D3', 'D1', 'D2', 'D3', 'D4', 'D1', 'D2', 'D3', 'D3', 'D1', 'D2', 'D2', 'D1', 'D1', 'D2', 'D3', 'D4', 'D4', 'D4', 'D3'], 'mean_score': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3.4, 2.6, 2.8667, 2.8, 2.4, 4.2, 4, 2.6, 3.4, 3.1333, 6.0667, 4.0667, 3.8, 3.5333, 4, 4.2, 5.8, 5.6, 4, 3.6, 3.9333, 5.2667, 5.2, 4.8, 3.9333, 4.6, 4.6, 5.2667, 5.9333, 3.8, 3.8, 2.4667, 3.2, 5.2, 3.8], 'grade': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 5, 3, 3, 3, 3, 7, 5, 3, 3, 3, 5, 5, 5, 3, 3, 3, 5, 5, 5, 3, 5, 5, 5, 5, 3, 3, 3, 3, 5, 3], 'PDI': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 33.333, 37.778, 28.889, 31.852, 31.111, 26.667, 46.667, 44.444, 28.889, 37.778, 34.815, 67.407, 45.185, 42.222, 39.259, 44.444, 46.667, 64.444, 62.222, 44.444, 40, 43.704, 58.519, 57.778, 53.333, 43.704, 51.111, 51.111, 58.519, 65.926, 42.222, 42.222, 27.407, 35.556, 57.778, 42.222]})
MODEL = {"task": "C", "model": "XGBoost", "why": "best overall and best ML on the leaves", "variant": "direct", "uav_metric": "R2", "uav_value": -0.1127193409562423, "uav_auc": None, "file": "C_XGBoost.npz", "kind": "xgb", "features": ["FD1090", "FD1060", "FD870", "FD595", "MCARI", "Dr"]}
EXPECTED = [2.209183, 2.514191, 2.286079, 1.854475, 2.790873, 2.524873, 2.339098, 3.560809, 2.896771, 1.827912, 2.870884, 2.809797, 1.974827, 1.630756, 2.383191, 2.405436, 1.815733, 2.588126, 2.509863, 2.755686, 2.16085, 2.339098, 2.943101, 2.943101, 2.646797, 1.953017, 2.263916, 2.391958, 3.583213, 2.126366, 2.265552, 4.193283, 3.0833, 2.943101, 2.094151, 2.772026, 2.082396, 2.943101, 2.701328, 2.790873, 2.484134, 2.484134, 2.250667, 2.281436, 2.917932, 2.365741, 2.129346, 3.643649]
GRADES = [0, 1, 3, 5, 7, 9]; KEY, TASK, VARIANT = 'C_XGBoost', MODEL['task'], MODEL['variant']
assert len(REF_WL) == len(WL) and np.abs(REF_WL - WL).max() < 0.01, 'the image wavelengths differ from the 24 Feb image used to build the model inputs'
GAIN = ASD_HEALTHY / UAV_HEALTHY; TARGET_MEAN = ASD_HEALTHY.mean()
print(f"{MODEL['model']} – task {TASK}, input '{VARIANT}', accuracy on the 48 UAV plots: {MODEL['uav_metric']} {MODEL['uav_value']:.3f}")

## 4 Features – identical to the ASD pipeline
For every pixel: (1) the input variant; (2) resampling to 1 nm, 350–2500 nm (gaps bridged linearly); (3) Savitzky–Golay smoothing / first derivative (window 11, order 2; Savitzky and Golay, 1964); (4) the model's features: first derivatives FD at given wavelengths, red-edge position REP and amplitude Dr (680–760 nm; Horler et al., 1983; Filella and Peñuelas, 1994), DWSI-2 = R1660/R550 (Apan et al., 2004), MCARI (Daughtry et al., 2000). Steps 2–3 are linear and are combined into one matrix evaluated only at the needed wavelengths – the same values as the ASD code, fast and light on memory.

In [ ]:
FULL = np.arange(350, 2501).astype(float)
_I = np.eye(len(WL)); MI = np.stack([np.interp(FULL, WL, _I[j]) for j in range(len(WL))], 1)   # (2151, bands): identical to np.interp per pixel
C0, C1 = savgol_coeffs(11, 2, deriv=0, use='dot'), savgol_coeffs(11, 2, deriv=1, use='dot')
def _op(w, c):
    v = np.zeros(len(FULL)); i = int(round(w - 350)); v[i - 5:i + 6] = c; return v
def build_operator(names):
    cols, keys = [], []
    def need(key, w, c):
        if key not in keys: cols.append(_op(w, c)); keys.append(key)
    for n in names:
        if n.startswith('FD') and n[2:].isdigit(): need(n, int(n[2:]), C1)
        elif n.startswith('R') and n[1:].isdigit(): need(n, int(n[1:]), C0)
        elif n in ('REP', 'Dr'):
            for w in range(680, 761): need('FD%d' % w, w, C1)
        elif n == 'DWSI-2': need('R1660', 1660, C0); need('R550', 550, C0)
        elif n == 'MCARI': need('R700', 700, C0); need('R670', 670, C0); need('R550', 550, C0)
        else: raise ValueError('feature not implemented: ' + n)
    return MI.T @ np.column_stack(cols), keys
def make_variant(S, v):
    if v == 'direct': return S
    C = S * GAIN
    return C if v == 'corrected' else C * (TARGET_MEAN / C.mean(1, keepdims=True))
def features(S, names, OP):
    M, keys = OP; Y = S.astype(np.float64) @ M; c = {k: j for j, k in enumerate(keys)}; g = lambda k: Y[:, c[k]]; out = []
    for n in names:
        if n in ('REP', 'Dr'):
            red = Y[:, [c['FD%d' % w] for w in range(680, 761)]]
            out.append(680 + red.argmax(1).astype(float) if n == 'REP' else red.max(1) * 1000)
        elif n == 'DWSI-2': out.append(g('R1660') / g('R550'))
        elif n == 'MCARI': out.append(((g('R700') - g('R670')) - 0.2 * (g('R700') - g('R550'))) * (g('R700') / g('R670')))
        else: out.append(g(n))
    return np.column_stack(out)

## 5 Load the model
Arrays from the `.npz` file; the prediction below reproduces scikit-learn/xgboost exactly. Inputs are standardised with the training mean and SD stored in the file.

In [ ]:
z = np.load(MODEL_FILE, allow_pickle=False); FEATURES = [str(f) for f in z['features']]; MU, SD = z['mu'], z['sd']
FEAT, THR, YES, NO, LEAF, ISLEAF, OFFSET, BASE = z['feat'], z['thr'], z['yes'], z['no'], z['leaf'], z['isleaf'], z['offset'], float(z['base'][0])
def predict(S):
    # spectra -> severity score: base score + sum of the leaf values of all boosted trees (XGBoost), clipped to 0-9
    Z = ((features(make_variant(S, VARIANT), FEATURES, OP) - MU) / SD).astype(np.float32); n = len(Z); rows = np.arange(n); v = np.full(n, BASE)
    for o in OFFSET:
        node = np.zeros(n, int)
        while True:
            g = o + node; lf = ISLEAF[g]
            if lf.all(): break
            go = Z[rows, FEAT[g]] < THR[g]; node = np.where(lf, node, np.where(go, YES[g], NO[g]))
        v += LEAF[o + node]
    v = np.clip(v, 0, 9); return v, v
print(f'XGBoost: {len(OFFSET)} trees, base score {BASE:.4f}; features: {", ".join(FEATURES)}')
OP = build_operator(FEATURES)

## 6 Self-check on the 48 plot-mean spectra
The model predicts the 48 plot means of the ENVI plot-average file; the result must agree with `EXPECTED` (computed when this notebook was built). A maximum difference below 0.001 means your computer reproduces the analysis exactly.

In [ ]:
if PLOT_AVG_FILE and os.path.exists(PLOT_AVG_FILE):
    L = open(PLOT_AVG_FILE).read().splitlines(); nh = sum(l.startswith('Column') for l in L)
    A = np.array([[float(x) for x in l.split()] for l in L[nh + 1:] if l.strip()])
    assert np.abs(A[:, 0] - WL).max() < 0.01, 'plot-average wavelengths differ from the image'
    d = np.abs(predict(A[:, 1:].T.astype(np.float32))[0] - np.array(EXPECTED)).max()
    print(f'self-check: max difference {d:.2e}', '-> OK' if d < 1e-3 else '-> CHECK library versions')
else:
    print('self-check skipped (no plot-average file)')

## 7 Open the image and build the crop mask
Pixels outside the field are 0 in the masked image. A pixel is used when its reflectance is valid and NDVI = (R898 − R684)/(R898 + R684) > `NDVI_MIN`. Reflectance is rescaled automatically if stored as 0–10000.

In [ ]:
ST = SPATIAL_STEP; NS, NL = len(range(0, cube.ns, ST)), len(range(0, cube.nl, ST))
bi = lambda w: int(np.argmin(np.abs(WL - w)))
sample = cube.rows(0, cube.nl, max(ST, cube.nl // 200))[:, ::max(1, cube.ns // 400), bi(800)]
v = sample[np.isfinite(sample) & (sample > 0)]
SCALE = (1e4 if np.percentile(v, 95) > 2 else 1.0) if REFLECTANCE_SCALE == 'auto' else float(REFLECTANCE_SCALE)
print(f'processing grid {NS} x {NL} (SPATIAL_STEP={ST}); reflectance divided by {SCALE:g}')

## 8 Pixel loop
The image is read `BLOCK_ROWS` rows at a time; for the crop pixels of each block the model computes its input, features and prediction. Values are NaN outside the crop.

In [ ]:
VAL = np.full((NL, NS), np.nan, np.float32); LAB = np.full((NL, NS), np.nan, np.float32); CROP = np.zeros((NL, NS), bool); t0 = time.time(); n_crop = 0
for r0 in range(0, cube.nl, BLOCK_ROWS * ST):
    r1 = min(cube.nl, r0 + BLOCK_ROWS * ST); blk = cube.rows(r0, r1, ST) / SCALE
    nd = (blk[..., bi(898)] - blk[..., bi(684)]) / (blk[..., bi(898)] + blk[..., bi(684)] + 1e-12)
    valid = np.isfinite(blk).all(2) & (blk[..., bi(800)] > 0) & (blk.mean(2) > 0) & (nd > NDVI_MIN)
    i0 = r0 // ST; nr = blk.shape[0]
    if valid.any():
        S = blk[valid]; CROP[i0:i0 + nr][valid] = True; n_crop += len(S)
        v, l = predict(S); VAL[i0:i0 + nr][valid] = v; LAB[i0:i0 + nr][valid] = l
    if (r0 // (BLOCK_ROWS * ST)) % 20 == 0 or r1 == cube.nl:
        el = time.time() - t0; print(f'rows {r1}/{cube.nl}  crop pixels {n_crop:,}  {el / 60:.1f} min  ETA {el / r1 * (cube.nl - r1) / 60:.1f} min')
base = os.path.join(OUT, KEY)
write_envi(base + '.dat', VAL, 'severity score (0-9)', 4, {'data ignore value': 'NaN'}); tif = write_tif(base + '.tif', VAL, np.nan)
print(f'done: {n_crop:,} crop pixels in {(time.time() - t0) / 60:.1f} min; GeoTIFF written: {bool(tif)}')

## 9 Map
Plot outlines and numbers are drawn from the ROI file. The title carries the model's accuracy on the 48 UAV plots so that an unreliable map is recognised at once.

In [ ]:
PLOTS = read_roi_xml(ROI_FILE) if ROI_FILE and os.path.exists(ROI_FILE) else {}
def outline(ax, numbers=True):
    for p, polys in PLOTS.items():
        for pg in polys: ax.plot((pg[:, 0] - 0.5) / ST, (pg[:, 1] - 0.5) / ST, color='k', lw=0.4)
        if numbers: c = (np.vstack(polys).mean(0) - 0.5) / ST; ax.text(c[0], c[1], str(p), fontsize=5, ha='center', va='center')
tag = f"UAV {MODEL['uav_metric'].replace('macro_F1', 'macro-F1').replace('R2', 'R²')} {MODEL['uav_value']:.2f}"
fig, axs = plt.subplots(1, 1, figsize=(8 * 1 + 1, 8 * NL / NS + 1), squeeze=False); axs = axs.ravel(); ax = axs[0]
im = ax.imshow(VAL, cmap='YlOrRd', vmin=0, vmax=9, interpolation='nearest'); lab = 'score (0-9)'
ttl = f"{MODEL['model']} severity score ({VARIANT} input; {tag}; NOT validated)"
ax.set_xlim(-0.5, NS - 0.5); ax.set_ylim(NL - 0.5, -0.5); outline(ax); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(ttl, fontsize=9); plt.colorbar(im, ax=ax, fraction=0.03, label=lab)
fig.tight_layout(); fp = os.path.join(OUT, KEY + '_map.png'); fig.savefig(fp, dpi=300); plt.close(fig); show(fp)

## 10 Per-plot accuracy
For each plot ROI: crop pixels inside the image and the plot value of the map, compared with the ASD leaf scores of 24 Feb. Score: mean predicted score vs the plot mean leaf score (R², RMSE, Pearson r). Plots 1 and 48 extend slightly beyond the image (95% and 89% of the plot inside); only their pixels inside the image are used.

In [ ]:
if PLOTS:
    yy, xx = np.mgrid[0:NL, 0:NS]; cx, cy = xx * ST + 0.5, yy * ST + 0.5; rows = []
    for p, polys in PLOTS.items():
        mk = np.zeros((NL, NS), bool)
        for pg in polys:
            sel = (cx >= pg[:, 0].min()) & (cx <= pg[:, 0].max()) & (cy >= pg[:, 1].min()) & (cy <= pg[:, 1].max())
            if sel.any(): mk[sel] |= MplPath(pg).contains_points(np.column_stack([cx[sel], cy[sel]]))
        mk &= CROP; v, l = VAL[mk], LAB[mk]; r = dict(plot=p, crop_pixels=int(mk.sum()))
        r.update(mean_score_pred=float(np.nanmean(v)) if len(v) else np.nan)
        rows.append(r)
    PS = pd.DataFrame(rows).merge(TRUTH, on='plot', how='left'); PS = PS[PS.crop_pixels > 0]
    e = PS.mean_score_pred - PS.mean_score
    S_ = dict(R2=1 - (e ** 2).sum() / ((PS.mean_score - PS.mean_score.mean()) ** 2).sum(), RMSE=float(np.sqrt((e ** 2).mean())), r_with_score=PS[['mean_score_pred', 'mean_score']].corr().iloc[0, 1])
    PS.to_csv(os.path.join(OUT, KEY + '_per_plot.csv'), index=False); pd.DataFrame([dict(model=KEY, plots=len(PS), **S_)]).to_csv(os.path.join(OUT, KEY + '_accuracy.csv'), index=False)
    print(PS.head(12).round(3).to_string(index=False)); print('...'); print(pd.Series(S_).round(3).to_string())
else:
    print('per-plot accuracy skipped (no ROI file)')

## 11 Outputs
In the results folder: the rasters (`C_XGBoost` (predicted severity score 0–9)), `C_XGBoost_map.png`, `C_XGBoost_per_plot.csv`, `C_XGBoost_accuracy.csv` and `run_info.json`. Open the `.dat/.hdr` or `.tif` in ENVI/QGIS – they carry the georeference of the image.

**References.** Apan et al. (2004); Barnes et al. (1989); Congalton (1991); Daughtry et al. (2000); Filella and Peñuelas (1994); Horler et al. (1983); Savitzky and Golay (1964); Sharma et al. (2019); Smith and Milton (1999); Westfall and Young (1993); Woo et al. (2023) – full entries in the transfer report.

In [ ]:
info = dict(model=KEY, image=DATA_FILE, spatial_step=ST, grid=[NS, NL], crop_pixels=int(n_crop), input=VARIANT, features=FEATURES, ndvi_min=NDVI_MIN, reflectance_scale=SCALE, minutes=round((time.time() - t0) / 60, 1))
json.dump(info, open(os.path.join(OUT, 'run_info.json'), 'w'), indent=1)
print('results in', OUT); print('\n'.join(sorted(os.listdir(OUT))))